# Who said what — MDE Design Freeze meeting
Speaker-attributed transcript (Whisper large-v3 + pyannote), with a refined pass on the Shailendra Q&A window.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Copy `MDE_Design Freeze.M4A` into Google Drive (e.g. `MyDrive/MDE/`).
3. Hugging Face: create a token, and accept the terms on **pyannote/speaker-diarization-community-1** and **pyannote/wespeaker-voxceleb-resnet34-LM**.
4. Add the token in Colab: 🔑 *Secrets* (left sidebar) → name `HF_TOKEN` → enable notebook access.

Run the cells top to bottom. Step 3 stops for you to name the speakers by ear.

In [1]:
# 0. Check the GPU (should show a Tesla T4 or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# 1. Install (takes ~2 min). If Colab asks to restart the session, restart and continue from the next cell.
!pip install -q faster-whisper pyannote.audio scipy soundfile
!ffmpeg -version | head -1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.4 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


In [4]:
# 2. Settings (no Google Drive)
import os
WORK = "/content/MDE"
os.makedirs(WORK, exist_ok=True)
OUT = f"{WORK}/analysis"
ENROLL_ARG = ""

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")
print("Ready. Work folder:", WORK)

Hugging Face token: ··········
Ready. Work folder: /content/MDE


In [5]:
   # Upload the M4A from your laptop (a file picker opens)
   from google.colab import files
   import shutil
   up = files.upload()                      # choose MDE_Design Freeze.M4A
   name = next(iter(up))
   AUDIO = f"{WORK}/{name}"
   shutil.move(name, AUDIO)
   print("Using:", AUDIO)

Saving MDE_Design Freeze.m4a to MDE_Design Freeze.m4a
Using: /content/MDE/MDE_Design Freeze.m4a


In [6]:
%%writefile /content/who_said_what.py
#!/usr/bin/env python
"""
who_said_what.py - speaker-attributed transcript for a multi-speaker meeting
recording, plus a focused high-precision pass over one discussion window.

Stages (run in order; each caches its output in the analysis folder):

  run     M4A -> 16 kHz mono WAV -> Whisper transcript (word timestamps)
          -> pyannote diarization -> each word assigned to a speaker
          -> per-speaker sample clips + speaker_map.json for you to fill in
  find    keyword search over the transcript to locate the discussion
  render  apply names from speaker_map.json; write full transcript
          (+ an excerpt for --focus START-END)
  refine  re-diarize + re-transcribe ONLY the focus window with tighter
          settings, carry names over by voice similarity, export one clip
          per turn so you can audit every attribution by ear
"""
import argparse
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import soundfile as sf

SR = 16000

DEFAULT_PROMPT = (
    "MDE design freeze meeting. Speakers: Harika, Suresh, Shailendra, Kevin, "
    "Nidhi, Uma. Topics: design approach, configuration build architecture, "
    "SE Gateway, IMS, devices, reuse across all devices."
)

DEFAULT_KEYWORDS = [
    "gateway", "ims", "all the devices", "all devices", "across all",
    "across the devices", "configuration", "build architecture",
    "design approach", "confirm", "reuse", "reusable", "common", "generic",
]


# ----------------------------------------------------------------- helpers
def ts(sec: float) -> str:
    sec = max(0.0, float(sec))
    h, rem = divmod(int(sec), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def srt_ts(sec: float) -> str:
    ms = int(round((sec - int(sec)) * 1000))
    return f"{ts(sec)},{ms:03d}"


def parse_ts(text: str) -> float:
    parts = [float(p) for p in text.strip().split(":")]
    total = 0.0
    for p in parts:
        total = total * 60 + p
    return total


def parse_range(text: str):
    a, b = text.split("-", 1)
    return parse_ts(a), parse_ts(b)


def workdir(audio: Path, out: str | None) -> Path:
    d = Path(out) if out else audio.with_name(audio.stem.replace(" ", "_") + "_analysis")
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_json(obj, path: Path):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def ffmpeg_to_wav(src: Path, dst: Path, normalize: bool):
    if not shutil.which("ffmpeg"):
        sys.exit("ffmpeg not found on PATH (Colab: !apt-get install -y ffmpeg).")
    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(src),
           "-ac", "1", "-ar", str(SR)]
    if normalize:
        # gentle: remove rumble, even out near/far voices (helps ASR on room recordings)
        cmd += ["-af", "highpass=f=80,dynaudnorm=f=250:g=15"]
    cmd += ["-c:a", "pcm_s16le", str(dst)]
    subprocess.run(cmd, check=True)


def load_audio(path: Path, start: float | None = None, end: float | None = None) -> np.ndarray:
    info = sf.info(str(path))
    s = int((start or 0) * SR)
    e = int(end * SR) if end is not None else info.frames
    data, sr = sf.read(str(path), start=s, stop=min(e, info.frames), dtype="float32")
    assert sr == SR, f"expected {SR} Hz, got {sr}"
    return data if data.ndim == 1 else data.mean(axis=1)


def hf_token() -> str:
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if not tok:
        sys.exit("Set HF_TOKEN (Hugging Face token; accept the pyannote model terms first).")
    return tok


def from_pretrained(cls, name: str):
    tok = hf_token()
    try:  # pyannote.audio >= 4
        return cls.from_pretrained(name, token=tok)
    except TypeError:  # pyannote.audio 3.x
        return cls.from_pretrained(name, use_auth_token=tok)


def torch_device():
    import torch
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------------------------------------ transcription
def transcribe(audio: np.ndarray, offset: float, model_name: str, prompt: str, beam: int):
    from faster_whisper import WhisperModel
    try:
        import ctranslate2
        cuda = ctranslate2.get_cuda_device_count() > 0
    except Exception:
        cuda = False
    device, compute = ("cuda", "float16") if cuda else ("cpu", "int8")
    print(f"[asr] {model_name} on {device}/{compute}")
    model = WhisperModel(model_name, device=device, compute_type=compute)
    segments, _ = model.transcribe(
        audio, language="en", beam_size=beam, word_timestamps=True,
        initial_prompt=prompt,
        condition_on_previous_text=False,   # avoids runaway repetition / hallucination
        vad_filter=True, vad_parameters={"min_silence_duration_ms": 500},
    )
    words = []
    total = len(audio) / SR
    for seg in segments:
        for w in seg.words or []:
            words.append({"start": round(w.start + offset, 3), "end": round(w.end + offset, 3),
                          "word": w.word, "prob": round(w.probability, 3)})
        print(f"\r[asr] {ts(seg.end)} / {ts(total)}", end="", flush=True)
    print()
    del model
    try:
        import gc, torch
        gc.collect(); torch.cuda.empty_cache()
    except Exception:
        pass
    return words


# --------------------------------------------------------------- diarization
def diarize(audio: np.ndarray, offset: float, model_name: str,
            num=None, min_spk=None, max_spk=None):
    import torch
    from pyannote.audio import Pipeline
    pipe = from_pretrained(Pipeline, model_name)
    pipe.to(torch_device())
    wav = {"waveform": torch.from_numpy(audio).unsqueeze(0), "sample_rate": SR}
    kw = {"num_speakers": num} if num else {k: v for k, v in
                                           (("min_speakers", min_spk), ("max_speakers", max_spk)) if v}
    print(f"[diar] {model_name} {kw}")
    try:
        from pyannote.audio.pipelines.utils.hook import ProgressHook
        with ProgressHook() as hook:
            out = pipe(wav, hook=hook, **kw)
    except ImportError:
        out = pipe(wav, **kw)
    regular = getattr(out, "speaker_diarization", out)
    exclusive = getattr(out, "exclusive_speaker_diarization", None) or regular

    def to_list(ann):
        return [{"start": round(t.start + offset, 3), "end": round(t.end + offset, 3), "speaker": spk}
                for t, _, spk in ann.itertracks(yield_label=True)]
    return to_list(regular), to_list(exclusive)


# ---------------------------------------------------- words -> speakers -> turns
def assign_speakers(words, regular, exclusive, max_gap=1.0):
    ex_s = np.array([s["start"] for s in exclusive]); ex_e = np.array([s["end"] for s in exclusive])
    ex_k = np.array([s["speaker"] for s in exclusive])
    rg_s = np.array([s["start"] for s in regular]); rg_e = np.array([s["end"] for s in regular])
    rg_k = np.array([s["speaker"] for s in regular])
    for w in words:
        ov = np.minimum(w["end"], ex_e) - np.maximum(w["start"], ex_s)
        spk = None
        if len(ov) and ov.max() > 0:
            pos = ov > 0
            totals = {}
            for k, o in zip(ex_k[pos], ov[pos]):
                totals[k] = totals.get(k, 0) + o
            spk = max(totals, key=totals.get)
        elif len(ov):
            mid = (w["start"] + w["end"]) / 2
            dist = np.minimum(np.abs(ex_s - mid), np.abs(ex_e - mid))
            if dist.min() <= max_gap:
                spk = ex_k[dist.argmin()]
        w["speaker"] = str(spk) if spk is not None else "UNKNOWN"
        mid = (w["start"] + w["end"]) / 2
        active = set(rg_k[(rg_s <= mid) & (rg_e >= mid)]) if len(rg_k) else set()
        w["overlap"] = len(active) > 1
    return words


def build_turns(words, gap=1.5):
    turns = []
    for w in words:
        if turns and turns[-1]["speaker"] == w["speaker"] and w["start"] - turns[-1]["end"] < gap:
            t = turns[-1]
            t["end"] = w["end"]; t["words"].append(w)
        else:
            turns.append({"speaker": w["speaker"], "start": w["start"], "end": w["end"], "words": [w]})
    for t in turns:
        ws = t.pop("words")
        t["text"] = "".join(x["word"] for x in ws).strip()
        t["overlap_frac"] = round(sum(x["overlap"] for x in ws) / len(ws), 2)
        t["lowconf_frac"] = round(sum(x["prob"] < 0.5 for x in ws) / len(ws), 2)
    return turns


# ------------------------------------------------------- voice embeddings
class Embedder:
    def __init__(self, model="pyannote/wespeaker-voxceleb-resnet34-LM"):
        from pyannote.audio import Inference, Model
        m = from_pretrained(Model, model)
        self.inf = Inference(m, window="whole")
        self.inf.to(torch_device())

    def vec(self, chunk: np.ndarray) -> np.ndarray:
        import torch
        v = np.asarray(self.inf({"waveform": torch.from_numpy(chunk).unsqueeze(0),
                                 "sample_rate": SR})).reshape(-1)
        return v / (np.linalg.norm(v) + 1e-9)

    def centroid(self, audio: np.ndarray, spans, base: float = 0.0, top=20, cap=10.0):
        spans = sorted(spans, key=lambda x: x[1] - x[0], reverse=True)[:top]
        vecs, wts = [], []
        for s, e in spans:
            s, e = s - base, min(e - base, s - base + cap)
            if e - s < 1.5:
                continue
            vecs.append(self.vec(audio[int(s * SR):int(e * SR)])); wts.append(e - s)
        if not vecs:
            return None
        c = np.average(np.stack(vecs), axis=0, weights=wts)
        return c / (np.linalg.norm(c) + 1e-9)


def clean_spans(exclusive, regular, speaker):
    """Exclusive segments of `speaker` that nobody else talks over."""
    others = [r for r in regular if r["speaker"] != speaker]
    out = []
    for s in exclusive:
        if s["speaker"] != speaker:
            continue
        if any(o["start"] < s["end"] and o["end"] > s["start"] for o in others):
            continue
        out.append((s["start"], s["end"]))
    return out


def match(rows: dict, cols: dict):
    """rows/cols: name -> unit vector. Returns similarity matrix + 1:1 assignment."""
    rk, ck = list(rows), list(cols)
    sim = np.array([[float(rows[r] @ cols[c]) for c in ck] for r in rk])
    try:
        from scipy.optimize import linear_sum_assignment
        ri, ci = linear_sum_assignment(-sim)
        pairs = {rk[i]: (ck[j], sim[i, j]) for i, j in zip(ri, ci)}
    except ImportError:
        pairs, used = {}, set()
        for i in np.argsort(-sim.max(axis=1)):
            for j in np.argsort(-sim[i]):
                if ck[j] not in used:
                    pairs[rk[i]] = (ck[j], sim[i, j]); used.add(ck[j]); break
    return rk, ck, sim, pairs


def print_matrix(rk, ck, sim):
    print("            " + "".join(f"{c[:10]:>11}" for c in ck))
    for i, r in enumerate(rk):
        print(f"{r[:11]:<12}" + "".join(f"{sim[i, j]:>11.2f}" for j in range(len(ck))))


# ------------------------------------------------------------------ stages
def stage_run(a):
    audio_path = Path(a.audio); wd = workdir(audio_path, a.out)
    clean_wav, asr_wav = wd / "audio_clean.wav", wd / "audio_asr.wav"
    if a.force or not clean_wav.exists():
        print("[prep] converting"); ffmpeg_to_wav(audio_path, clean_wav, normalize=False)
        ffmpeg_to_wav(audio_path, asr_wav, normalize=True)

    words_p, diar_p = wd / "words.json", wd / "diarization.json"
    if a.force or not words_p.exists():
        save_json(transcribe(load_audio(asr_wav), 0.0, a.model, a.prompt, a.beam), words_p)
    if a.force or not diar_p.exists():
        reg, exc = diarize(load_audio(clean_wav), 0.0, a.diar_model,
                           a.speakers, a.min_speakers, a.max_speakers)
        save_json({"regular": reg, "exclusive": exc}, diar_p)

    words, diar = load_json(words_p), load_json(diar_p)
    words = assign_speakers(words, diar["regular"], diar["exclusive"])
    turns = build_turns(words)
    save_json(turns, wd / "turns.json")

    # per-speaker samples for labeling by ear
    audio = load_audio(clean_wav)
    sdir = wd / "speaker_samples"; sdir.mkdir(exist_ok=True)
    speakers = sorted({s["speaker"] for s in diar["exclusive"]})
    talk = {k: sum(s["end"] - s["start"] for s in diar["exclusive"] if s["speaker"] == k) for k in speakers}
    print("\n[label] speakers found (talk time):")
    emb = Embedder(); cents = {}
    for k in sorted(speakers, key=talk.get, reverse=True):
        spans = clean_spans(diar["exclusive"], diar["regular"], k) or \
            [(s["start"], s["end"]) for s in diar["exclusive"] if s["speaker"] == k]
        best = sorted(spans, key=lambda x: x[1] - x[0], reverse=True)[:3]
        gap = np.zeros(int(0.6 * SR), dtype=np.float32); parts = []
        for s, e in best:
            parts += [audio[int(s * SR):int(min(e, s + 12) * SR)], gap]
        if parts:
            sf.write(sdir / f"{k}.wav", np.concatenate(parts), SR)
        c = emb.centroid(audio, spans)
        if c is not None:
            cents[k] = c
        lines = [t for t in sorted(turns, key=lambda t: t["end"] - t["start"], reverse=True)
                 if t["speaker"] == k][:2]
        print(f"  {k}  {talk[k] / 60:5.1f} min  clips @ " + ", ".join(ts(s) for s, _ in best))
        for t in lines:
            print(f"      [{ts(t['start'])}] {t['text'][:160]}")
    np.savez(wd / "centroids.npz", **cents)

    # optional: auto-suggest names from enrollment clips (enroll/Harika.wav, ...)
    smap = {k: "" for k in speakers}
    if a.enroll and Path(a.enroll).is_dir():
        edir = wd / "enroll_16k"; edir.mkdir(exist_ok=True); refs = {}
        for f in sorted(Path(a.enroll).iterdir()):
            if f.suffix.lower() not in {".wav", ".m4a", ".mp3", ".flac", ".ogg", ".mp4"}:
                continue
            dst = edir / (f.stem + ".wav"); ffmpeg_to_wav(f, dst, normalize=False)
            ea = load_audio(dst); n = len(ea) / SR
            c = emb.centroid(ea, [(s, min(s + 10, n)) for s in np.arange(0, n, 10)], top=50)
            if c is not None:
                refs[f.stem] = c
        if refs and cents:
            rk, ck, sim, pairs = match(cents, refs)
            print("\n[enroll] cosine similarity (diarized speaker x enrolled person):")
            print_matrix(rk, ck, sim)
            for k, (name, s) in pairs.items():
                row = sim[rk.index(k)]; second = np.sort(row)[-2] if len(row) > 1 else 0
                ok = s >= a.min_sim and s - second >= 0.05
                print(f"  {k} -> {name:<12} sim={s:.2f} margin={s - second:.2f} {'' if ok else '(weak - verify)'}")
                if ok:
                    smap[k] = name
    mp = wd / "speaker_map.json"
    if mp.exists() and not a.force:
        print(f"\n[label] kept existing {mp}")
    else:
        save_json(smap, mp)
    print(f"\nNext: listen to {sdir}/*.wav, fill names in {mp}"
          "\n      (map two labels to the same name if one person was split).")


def names_for(wd: Path, fname="speaker_map.json"):
    p = wd / fname
    return {k: v for k, v in load_json(p).items() if v} if p.exists() else {}


def write_outputs(turns, names, base: Path, title: str):
    def nm(t): return names.get(t["speaker"], t["speaker"])
    md = [f"# {title}", "", "Flags: ⚠ = overlapping speech, ? = many low-confidence words", ""]
    srt, csv = [], ["start,end,speaker,raw_label,overlap_frac,lowconf_frac,text"]
    for i, t in enumerate(turns, 1):
        flag = (" ⚠" if t["overlap_frac"] > 0.3 else "") + (" ?" if t["lowconf_frac"] > 0.3 else "")
        md.append(f"**[{ts(t['start'])}] {nm(t)}**{flag}: {t['text']}\n")
        srt += [str(i), f"{srt_ts(t['start'])} --> {srt_ts(t['end'])}", f"{nm(t)}: {t['text']}", ""]
        txt = t["text"].replace('"', '""')
        csv.append(f'{ts(t["start"])},{ts(t["end"])},{nm(t)},{t["speaker"]},'
                   f'{t["overlap_frac"]},{t["lowconf_frac"]},"{txt}"')
    talk = {}
    for t in turns:
        talk[nm(t)] = talk.get(nm(t), 0) + t["end"] - t["start"]
    md += ["", "## Talk time", ""] + [f"- {k}: {v / 60:.1f} min" for k, v in
                                      sorted(talk.items(), key=lambda x: -x[1])]
    base.with_suffix(".md").write_text("\n".join(md), encoding="utf-8")
    base.with_suffix(".srt").write_text("\n".join(srt), encoding="utf-8")
    base.with_suffix(".csv").write_text("\n".join(csv), encoding="utf-8-sig")
    print(f"[out] {base}.md / .srt / .csv")


def stage_render(a):
    wd = workdir(Path(a.audio), a.out)
    turns, names = load_json(wd / "turns.json"), names_for(wd)
    write_outputs(turns, names, wd / "transcript", "Full transcript")
    if a.focus:
        s, e = parse_range(a.focus)
        sel = [t for t in turns if t["end"] >= s and t["start"] <= e]
        write_outputs(sel, names, wd / "focus", f"Focus {ts(s)}-{ts(e)}")


def stage_find(a):
    wd = workdir(Path(a.audio), a.out)
    turns, names = load_json(wd / "turns.json"), names_for(wd)
    kws = [k.strip().lower() for k in (a.keywords.split(",") if a.keywords else DEFAULT_KEYWORDS)]
    hits = []
    for t in turns:
        low = re.sub(r"[^\w\s]", " ", t["text"].lower())
        found = [k for k in kws if k in low]
        if found:
            hits.append((t, found))
            print(f"[{ts(t['start'])}] {names.get(t['speaker'], t['speaker'])}: "
                  f"{{{', '.join(found)}}} {t['text'][:150]}")
    if not hits:
        print("no hits - try --keywords with variants Whisper may have heard"); return
    win = a.window * 60
    best = max(hits, key=lambda h: sum(len(x[1]) for x in hits
                                       if h[0]["start"] <= x[0]["start"] < h[0]["start"] + win))
    s = best[0]["start"]
    print(f"\nDensest {a.window}-min window starts ~{ts(s)}. Suggested: "
          f"--focus {ts(max(0, s - 60))}-{ts(s + win)}")


def stage_refine(a):
    wd = workdir(Path(a.audio), a.out)
    start, end = parse_ts(a.start), parse_ts(a.end)
    pad = 2.0; s0 = max(0.0, start - pad)
    fdir = wd / f"focus_{ts(start).replace(':', '')}_{ts(end).replace(':', '')}"
    fdir.mkdir(exist_ok=True)
    turns_p = fdir / "turns.json"

    if not (a.rerender and turns_p.exists()):
        clean = load_audio(wd / "audio_clean.wav", s0, end + pad)
        asr = load_audio(wd / "audio_asr.wav", s0, end + pad)
        words = transcribe(asr, s0, a.model, a.prompt, a.beam)
        kw = {"num": a.speakers} if a.speakers else {"min_spk": 2, "max_spk": 7}
        reg, exc = diarize(clean, s0, a.diar_model, **kw)
        save_json({"regular": reg, "exclusive": exc}, fdir / "diarization.json")
        words = [w for w in assign_speakers(words, reg, exc) if start <= w["start"] <= end]
        turns = build_turns(words, gap=1.0)
        save_json(turns, turns_p)

        # carry names over: refined cluster voice -> full-run cluster voice -> name
        full = dict(np.load(wd / "centroids.npz")) if (wd / "centroids.npz").exists() else {}
        names = names_for(wd)
        emb = Embedder(); cents = {}
        for k in sorted({x["speaker"] for x in exc}):
            spans = clean_spans(exc, reg, k) or [(x["start"], x["end"]) for x in exc if x["speaker"] == k]
            c = emb.centroid(clean, spans, base=s0)
            if c is not None:
                cents[k] = c
        fmap = {k: "" for k in cents}
        if full and cents:
            # collapse full-run labels that share a name into one voice per person
            per_name = {}
            for lab, v in full.items():
                per_name.setdefault(names.get(lab, lab), []).append(v)
            people = {n: (np.mean(v, 0) / np.linalg.norm(np.mean(v, 0))) for n, v in per_name.items()}
            rk, ck, sim, pairs = match(cents, people)
            print("\n[refine] focus-window speaker x person similarity:")
            print_matrix(rk, ck, sim)
            for k, (n, s) in pairs.items():
                fmap[k] = n if s >= a.min_sim else ""
                print(f"  {k} -> {n} ({s:.2f}){'' if s >= a.min_sim else '  weak - fill manually'}")
        save_json(fmap, fdir / "speaker_map.json")

        # one clip per turn -> audit every attribution by ear
        cdir = fdir / "turn_clips"; cdir.mkdir(exist_ok=True)
        for i, t in enumerate(turns, 1):
            seg = clean[int((t["start"] - s0) * SR):int((t["end"] - s0) * SR)]
            name = fmap.get(t["speaker"]) or t["speaker"]
            sf.write(cdir / f"{i:03d}_{ts(t['start']).replace(':', '-')}_{name}.wav", seg, SR)

    turns = load_json(turns_p)
    write_outputs(turns, names_for(fdir), fdir / "focus_refined",
                  f"Focus (refined) {ts(start)}-{ts(end)}")
    print(f"Fix any label in {fdir / 'speaker_map.json'}, then rerun with --rerender.")


# --------------------------------------------------------------------- CLI
def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)

    def common(sp):
        sp.add_argument("audio"); sp.add_argument("--out", help="analysis folder (default: next to audio)")

    def models(sp, beam):
        sp.add_argument("--model", default="large-v3", help="Whisper model (large-v3 | large-v3-turbo | distil-large-v3)")
        sp.add_argument("--beam", type=int, default=beam)
        sp.add_argument("--prompt", default=DEFAULT_PROMPT, help="names + jargon to bias spelling")
        sp.add_argument("--diar-model", default="pyannote/speaker-diarization-community-1",
                        help="or pyannote/speaker-diarization-3.1 on pyannote.audio 3.x")
        sp.add_argument("--min-sim", type=float, default=0.45, help="min cosine sim to auto-name")

    r = sub.add_parser("run"); common(r); models(r, 5)
    r.add_argument("--speakers", type=int, help="exact count if you are sure")
    r.add_argument("--min-speakers", type=int, default=5)
    r.add_argument("--max-speakers", type=int, default=8)
    r.add_argument("--enroll", help="folder with Name.wav/.m4a voice samples (10-60 s each)")
    r.add_argument("--force", action="store_true")
    r.set_defaults(fn=stage_run)

    f = sub.add_parser("find"); common(f)
    f.add_argument("--keywords", help="comma-separated; default covers gateway/IMS/devices/etc.")
    f.add_argument("--window", type=int, default=10, help="minutes")
    f.set_defaults(fn=stage_find)

    d = sub.add_parser("render"); common(d)
    d.add_argument("--focus", help="HH:MM:SS-HH:MM:SS")
    d.set_defaults(fn=stage_render)

    x = sub.add_parser("refine"); common(x); models(x, 8)
    x.add_argument("--start", required=True); x.add_argument("--end", required=True)
    x.add_argument("--speakers", type=int, help="people who talk in this window (e.g. 4)")
    x.add_argument("--rerender", action="store_true", help="only re-apply edited speaker_map.json")
    x.set_defaults(fn=stage_refine)

    a = p.parse_args(); a.fn(a)


if __name__ == "__main__":
    main()

Writing /content/who_said_what.py


## Stage 1 — Full pass: transcribe + diarize
~20–40 min for a 1-hour meeting on a T4. Outputs are cached in `OUT`, so if Colab disconnects, rerun cells 1, 2, the `%%writefile` cell, and this one; finished steps are skipped.
For a faster first pass, add `--model large-v3-turbo`.

In [9]:
# Fix: CUDA 12 libraries for faster-whisper
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
import os, nvidia.cublas.lib, nvidia.cudnn.lib
libs = [nvidia.cublas.lib.__path__[0], nvidia.cudnn.lib.__path__[0]]
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("OK:", libs)

OK: ['/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib', '/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib']


In [10]:
!python /content/who_said_what.py run "{AUDIO}" --out "{OUT}" --min-speakers 5 --max-speakers 8 {ENROLL_ARG}

[asr] large-v3 on cuda/float16
[asr] 00:30:08 / 00:30:10
config.yaml: 100% 444/444 [00:00<00:00, 2.36MB/s]

segmentation/pytorch_model.bin: downloading bytes:   0% 0.00/5.91M [00:00<?, ?B/s]
segmentation/pytorch_model.bin: downloading bytes: 100% 5.59M/5.59M [00:00<00:00, 9.34MB/s,  548kB/s  ]
segmentation/pytorch_model.bin: reconstructing file: 100% 5.91M/5.91M [00:00<00:00, 9.87MB/s,  580kB/s  ]

plda/xvec_transform.npz: downloading bytes:   0% 0.00/134k [00:00<?, ?B/s]
plda/xvec_transform.npz: downloading bytes: 100% 127k/127k [00:00<00:00, 316kB/s, 12.5kB/s  ]
plda/xvec_transform.npz: reconstructing file: 100% 134k/134k [00:00<00:00, 335kB/s, 13.3kB/s  ]

plda/plda.npz: downloading bytes:   0% 0.00/134k [00:00<?, ?B/s]
plda/plda.npz: downloading bytes: 100% 134k/134k [00:00<00:00, 328kB/s, 13.2kB/s  ]
plda/plda.npz: reconstructing file: 100% 134k/134k [00:00<00:00, 328kB/s, 13.2kB/s  ]

embedding/pytorch_model.bin: downloading bytes:   8% 2.11M/26.6M [00:00<00:05, 4.89MB/s]
embeddi

## Stage 2 — ⏸ YOU name the speakers
Listen to each clip below, then fill in the names in the next cell.
- If one person appears under two labels, give both the same name.
- Leave a label as `""` if you can't tell.

In [11]:
import json, glob
from pathlib import Path
from IPython.display import Audio, display, Markdown

turns = json.load(open(f"{OUT}/turns.json"))
smap  = json.load(open(f"{OUT}/speaker_map.json"))
for wav in sorted(glob.glob(f"{OUT}/speaker_samples/*.wav")):
    lab = Path(wav).stem
    display(Markdown(f"### {lab}  (suggested: {smap.get(lab) or '—'})"))
    for t in sorted([t for t in turns if t["speaker"] == lab], key=lambda t: t["start"] - t["end"])[:2]:
        print("   ", t["text"][:160])
    display(Audio(wav))
print("Current map:", json.dumps(smap, indent=1))

### SPEAKER_00  (suggested: —)

    no no no what we're saying Schillinger is that what we need to you to send to Nessie is standardized we'll do we'll do the interpretations for different operati
    two interfaces are in four channels are industry standard ip3 ipv4 and bb6 all that stuff is industry standard there's nothing that we're using here in the mode


### SPEAKER_01  (suggested: —)

    it's not going to be straightforward first of all like who has this holistic list of all routers and all all these things and they can quantify this is like we 
    other things sorry i was gonna go like yes harika is looking for this is bundled it's not evident from etf so we will have to talk back because uh it's all trai


### SPEAKER_02  (suggested: —)

    telling us it's a layer 2 connection meridian is telling us it's a layer 2 connection needs to be established and you're providing us what are the ports involve
    telling me that the protocol is BGP so then I on the Nessie side we conclude or infer from this information that we need to create BGP neighbors on the A side a


... (3 outputs left out)

    Is it at the logical resource assignment or at the network architecture based decision engine. So I think my assumption is that L2 and L3 that we are sending it
    and I mean like you know if in the payload in the L2 and L3 if we have more examples comments


### SPEAKER_05  (suggested: —)

    Yeah, I understand Nessie will handle all of that, and there's already months or years of history there. What I'm saying is it all starts at MDE in the new worl
    but uh but like in terms of these are standard fields these can be templated that don't differ much from sg gate with yms i feel like those things we should hav


Current map: {
 "SPEAKER_00": "",
 "SPEAKER_01": "",
 "SPEAKER_02": "",
 "SPEAKER_03": "",
 "SPEAKER_04": "",
 "SPEAKER_05": ""
}


In [12]:
# Fill in the names, then run this cell.
NAMES = {
    "SPEAKER_00": "Kevin",
    "SPEAKER_01": "Nidhi",
    "SPEAKER_02": "Harika",
    "SPEAKER_03": "Uma",
    "SPEAKER_04": "Suresh",
    "SPEAKER_05": "Shailendra",
}
smap = json.load(open(f"{OUT}/speaker_map.json"))
smap.update(NAMES)
json.dump(smap, open(f"{OUT}/speaker_map.json", "w"), indent=2)
print(json.dumps(smap, indent=1))

{
 "SPEAKER_00": "Kevin",
 "SPEAKER_01": "Nidhi",
 "SPEAKER_02": "Harika",
 "SPEAKER_03": "Uma",
 "SPEAKER_04": "Suresh",
 "SPEAKER_05": "Shailendra"
}


## Stage 3 — Find the Shailendra Q&A window

In [13]:
!python /content/who_said_what.py find "{AUDIO}" --out "{OUT}"

[00:00:21] Nidhi: {generic} But let's not go there yet, right? Like, I think I just want to maybe... So the payload builder is like, we are sending you a generic, not generic, bu
[00:00:51] Harika: {common} Yeah, so for instance, if we agree on this layer-to-connection format, Meridian is essentially telling us that, hey, this is the speed of an inter... 
[00:04:39] Harika: {gateway, ims} is a layer to connection. It should not differ from an SE gateway to an IMS or some other design. What changes from design to design, at least at our 
[00:04:59] Harika: {gateway} service has a rule saying the bundle should be within this range. Like for AC Gateway, we know that the bundle ID should be within 610 to 615, right? 
[00:05:57] Shailendra: {gateway, ims} question about what you said about layer 2 connection is a layer 2 connection it doesn't is there whether it's a C gateway or IMS there is identical t
[00:06:12] Harika: {ims} should be what changes is your certain parameters like IMS may 

Copy the suggested window from the last line above into `FOCUS_START` / `FOCUS_END`.

In [16]:
FOCUS_START = "00:00:00"
FOCUS_END   = "00:30:10"
FOCUS_SPEAKERS = 6         # Kevin, Nidhi, Harika, Uma, Suresh, Shailendra

!python /content/who_said_what.py render "{AUDIO}" --out "{OUT}" --focus {FOCUS_START}-{FOCUS_END}

[out] /content/MDE/analysis/transcript.md / .srt / .csv
[out] /content/MDE/analysis/focus.md / .srt / .csv


## Stage 4 — Refined pass on that window (exact speaker count, more thorough decoding)

In [17]:
!python /content/who_said_what.py refine "{AUDIO}" --out "{OUT}" --start {FOCUS_START} --end {FOCUS_END} --speakers {FOCUS_SPEAKERS}

[asr] large-v3 on cuda/float16
[asr] 00:30:08 / 00:30:10
[diar] pyannote/speaker-diarization-community-1 {'num_speakers': 6}
/usr/local/lib/python3.13/dist-packages/pyannote/audio/utils/reproducibility.py:
74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might 
lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
segmentation     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:03
segmentation     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:03
segmentation     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:03
/usr/local/lib/python3.13/dist-packages/pyannote/audio/models/blocks/pooling.py:
103: UserWarning: std(): degrees of freedom is <= 0. Correction should be 
strictly less than the reduction factor (input n

## Stage 5 — ⏸ Audit every turn by ear
Each turn has its own clip. Play any that look wrong (especially ⚠ / ? lines), fix the names in `FIX`, and rerun.

In [18]:
fdir = sorted(glob.glob(f"{OUT}/focus_*_*/"))[-1]
print("Focus folder:", fdir)
print(json.dumps(json.load(open(f"{fdir}/speaker_map.json")), indent=1))
for wav in sorted(glob.glob(f"{fdir}/turn_clips/*.wav")):
    print(Path(wav).stem); display(Audio(wav))

Focus folder: /content/MDE/analysis/focus_002300_004100/
{
 "SPEAKER_00": "Suresh",
 "SPEAKER_01": "Uma",
 "SPEAKER_02": "Harika",
 "SPEAKER_03": "Nidhi"
}
001_00-23-04_Suresh


002_00-23-18_Harika


003_00-23-33_Nidhi


004_00-23-35_Harika


005_00-23-36_Nidhi


006_00-23-48_Suresh


... (38 outputs left out)

045_00-29-53_UNKNOWN


046_00-29-54_Suresh


047_00-29-58_Harika


048_00-30-00_Suresh


In [21]:
fdir = f"{OUT}/focus_000000_003010"     # the full-meeting refined pass
FIX = {"SPEAKER_04": "Shailendra"}

fmap = json.load(open(f"{fdir}/speaker_map.json"))
fmap.update(FIX)
json.dump(fmap, open(f"{fdir}/speaker_map.json", "w"), indent=2)
!python /content/who_said_what.py refine "{AUDIO}" --out "{OUT}" --start 00:00:00 --end 00:30:10 --rerender

print(json.dumps(fmap, indent=1))
display(Markdown(open(f"{fdir}/focus_refined.md", encoding="utf-8").read()))

[out] /content/MDE/analysis/focus_000000_003010/focus_refined.md / .srt / .csv
Fix any label in /content/MDE/analysis/focus_000000_003010/speaker_map.json, then rerun with --rerender.
{
 "SPEAKER_00": "Kevin",
 "SPEAKER_01": "Nidhi",
 "SPEAKER_02": "Harika",
 "SPEAKER_03": "Uma",
 "SPEAKER_04": "Shailendra",
 "SPEAKER_05": "Shailendra"
}


# Focus (refined) 00:00:00-00:30:10

Flags: ⚠ = overlapping speech, ? = many low-confidence words

**[00:00:00] Harika**: This layer-to-connection format, Meridian is essentially telling us

**[00:00:04] Kevin**: what it is, the format, right? We need to come up with the contract between Meridian and Nessie.

**[00:00:12] Uma**: Oh, yeah. I'm not worrying about that. So, basically, like, based on the... You're talking about... Yeah. So, yeah.

**[00:00:21] Nidhi**: But let's not go there yet, right? Like, I think I just want to maybe... So the payload builder is like, we are sending you a generic, not generic, but like a coordinate request, but if it needs four elements, three elements, then you are converting that into individual, your input is kind of high level and then you are converting it to different service decompositions, I guess.

**[00:00:48] Kevin**: Yes, so Harika has examples there.

**[00:00:51] Harika**: Yeah, so for instance, if we agree on this layer-to-connection format, Meridian is essentially telling us that, hey, this is the speed of an inter... Like, we group the common parameters into one section. You're essentially saying, hey, this is the A-side device, this is the Z-side device.

**[00:01:09] Harika**: Ignore this, we'll talk about that later. And this is the VLAN that needs to be added on the connection.

**[00:01:16] Kevin**: And this is built based on rules in the carrot.

**[00:01:20] Nidhi** ?: that's right really

**[00:01:27] Kevin**: thinking about going forward we should just call it the carrot

**[00:01:30] Kevin**: you're telling

**[00:01:35] Harika**: us it's a layer to connection meridian is telling us it's a layer to connection needs to be established and you're providing us what are the ports involved the A-side V-side port and the A-side V-side devices so

**[00:01:49] Harika**: Nessie will take this input and convert it into since you

**[00:01:59] Harika**: said it's a bundle it's not a physical interface it all needs to be grouped together and you're giving us a speed vlan and what are the inter physical interfaces that need to go in there we would the payload builder here on the nessie side will understand that we need to create a l2 ether channel ad which is for creating a layer 2 bundle and then we need to do l2 interface and it's not listed here but that's another activity we will add to the stack and also l3 ether channel member ad because

**[00:02:32] UNKNOWN** ?: Because

**[00:02:32] Harika**: these three activities will together make up building a bundle layer to connect.

**[00:02:41] Harika**: And of course, we would need... Yeah, so,

**[00:02:42] Nidhi**: sorry. So, I guess my question is... Payload, I'm... Yeah, I think we are familiar with that, right? Like, we are sending you L2 and L3, right? All this information. And you are decomposing. But this map here, for example... Sister

**[00:03:02] Nidhi**: General, like, we are saying L2. We are not telling you this is an Ethernet channel add or this is a... because different routers call things so so what do you have to build on that is that like a pair which is configurable map of services for you is that's how

**[00:03:24] Kevin** ⚠ ?: that's part of our that's part of the rules and allocation like

**[00:03:30] Kevin**: the model data

**[00:03:36] Kevin**: is this is this a or is it a fortune right is it a prefix set or a prefix list because those things are different on different iOS devices also and once Once

**[00:03:47] UNKNOWN**: we,

**[00:03:47] Kevin** ?: you know, down the road, get into a developer

**[00:03:56] Kevin**: device.

**[00:04:05] Nidhi** ?: Managing as a

**[00:04:06] Kevin**: certain

**[00:04:32] Nidhi**: rules or is it just connection

**[00:04:39] Harika**: is a layer to connection. It should not differ from an SE gateway to an IMS or some other design. What changes from design to design, at least at our layer, maybe, is the allocation rules. As in, like, for layer to connections, when we create the bundle, the

**[00:04:59] Harika**: service has a rule saying the bundle should be within this range. Like for AC Gateway, we know that the bundle ID should be within 610 to 615, right? That rule is specific to AC Gateway. So

**[00:05:11] Harika**: those rules, if there is a specific rule for assignment for the runtime resources that Nessie assigns, we will maintain those. But other than that, this is just a layer 2 connection.

**[00:05:24] Nidhi**: Okay, so all I was trying to understand is those rules are like coded or those rules can change, are they configurable like the suggestions or? They

**[00:05:33] Kevin**: they are coded they are coded but also they can change

**[00:05:37] Harika**: Yeah, for our first two iterations. We want to hard code them, but our ultimate goal

**[00:05:42] UNKNOWN**: is to

**[00:05:44] UNKNOWN**: Expose them to engineer so they can control They

**[00:05:52] Harika**: can control existing question

**[00:05:57] Shailendra**: about what you said about layer 2 connection is a layer 2 connection It doesn't it is there where there's a C gateway or IMS there is

**[00:06:09] Shailendra** ?: identical they

**[00:06:12] Harika**: should be what changes is your certain parameters like IMS may not do a bundle they might be just doing a layer to our physical port.

**[00:06:22] Harika**: SA Gateway is doing it as a bundle.

**[00:06:29] Nidhi**: Think about from an in because we will model this into inventory models right so we shouldn't go to a specific if a configuration needs some change that's a domain orchestrator that needs to manage that okay for this device this is how you do it but our inventory should be little bit agnostic in the sense that layer 2 these are the parameters some are populated some are not populated depending on how

**[00:06:55] Shailendra**: the design goes i agree with that principle i'm just saying if if the hardware being deployed if if layer 2 the way layer 2 connection is configured is agnostic of the hardware being deployed then we can templatize this and that's a candidate for automation is how i'm looking at it

**[00:07:20] Harika**: that's what we are targeting

**[00:07:23] Harika** ?: these are we

**[00:07:25] Harika**: would need to configure a layer 2 connection cisco

**[00:07:30] Harika**: or any other device that we support today and

**[00:07:35] Harika**: this is our first strategy right maybe we might be missing one or two we will get to them as these uh

**[00:07:42] Harika**: as we move through the mvp phase or as we start testing we might see one or two but other than that as

**[00:07:52] Harika**: mentioned earlier this is all we would need right now to turn up

**[00:07:56] UNKNOWN**: a

**[00:07:56] Harika**: layer to port irrespective of who it is for

**[00:08:02] Harika**: similarly for a layer 3 connection

**[00:08:05] Harika**: This doesn't involve everything, I built this specifically for BGP, a layer 3 connection which is using a route, BGP routing protocol, we can extend this for OSPF and Stackroup. If

**[00:08:22] Harika**: we agree on this would become the layer

**[00:08:25] Harika**: 3 connection, and you are pretty much giving me information on which VRM I need to configure on, this key will help me decide if it needs to be a sub interface or just a logical bundle you're

**[00:08:48] Harika**: telling me that the protocol is pgp so then i on the nasi side we conclude or infer from this information that we need to create pgp neighbors on the a side and z side okay so those would be the rules or decision matrix on the And similarly, like you mentioned, Didi, would you maintain anything design specific? I think along with the rules for port channel IDs, the bundle IDs, we would also maintain

**[00:09:22] Harika**: rules saying if it is an SA gateway N3, whenever you create a BGP neighbor, you must assign this route policy, you must create this prefix list, that would be a general for the specific use case. So those would be coded for now, coded as a separate module for now, and then we'll work towards exposing them so HameChain then can control exactly what goes into that prefix list or what goes into that route now. Because end of the day they are standard configs they are creating for that design, which they should have control over to

**[00:09:59] Harika** ?: maintain. So

**[00:10:08] Shailendra**: Harika, does that mean these layer 2 and layer 3 config templates will be sent by Nessie beforehand to the Meridian Orchestrator for the Orchestrator to then pick and choose from the CIQ fields and populate these and then send down to Nessie?

**[00:10:31] Harika**: No, so this is what we will agree on in next couple of weeks, right? We would talk this through and agree on,

**[00:10:40] UNKNOWN**: hey,

**[00:10:40] Harika**: does this make sense for a layer-to-connection does this API spec accounts for everything, if we agree on then how...

**[00:10:48] Shailendra**: When you say we, is it just the audience here in this call or you need to run this by Klemchen or others?

**[00:10:55] Harika**: First it needs to be agreed by the audience in this call and then we can

**[00:11:02] Harika**: inform the business team on the overall setup and get their inputs on the

**[00:11:11] Shailendra**: that's where I see a disconnected do you think that we can agree on this just this audience and then progress further and then at what point were you thinking of involving business and engine once

**[00:11:28] Harika**: our solution talks are finalized and we say okay this looks like a good solution we can present to our end customers yeah

**[00:11:38] Shailendra**: and my concern with that approach is what if there are big gaps that they point out when we reach all that all the way there I personally think we should involve them here at this stage I don't know what you guys think

**[00:11:54] Harika**: Heng Chung and team doesn't have any input in how Meridian and Nessie should talk to right

**[00:12:00] Shailendra**: that's not what I'm talking about I'm talking about the standardization of these blocks that we're going on layer two and layer three there's I know you're aware of yeah

**[00:12:12] Uma** ⚠ ?: yeah with

**[00:12:12] Shailendra**: Nessie but but like in terms of these are standard fields these can be templated that don't differ much from Seagate with YMS I feel like those things we should have app business vet, and we have to be 100% sure that these assumptions are actually valid before we instrument this and take it and complete the solution design.

**[00:12:39] Harika**: Okay, maybe agree is not the right term. First, we need to be on the same page about the API spec, and then we can bring in customers saying, hey, this is what we're thinking we'll be doing. Does this align with your vision? Does this align with what you're planning

**[00:12:56] Harika** ?: when consuming an

**[00:12:58] Shailendra**: DE?

**[00:12:59] Shailendra**: For the API spec, I see that as just the mechanics, it's

**[00:13:03] UNKNOWN**: a

**[00:13:03] Shailendra**: structure, but I'm talking about the actual principle, the logic, like whatever we map out here, does it

**[00:13:10] Harika**: translate? Right, Shalendra, I cannot go to Henshend and show him this as a thought process and say, hey, do you agree to this? No, that's not what I'm suggesting. I'm saying we will build our solution docs and have the detailed components listed saying hey what is this component expected to do like for our customers we need to tell them how their user experience would be with the solution and what do they get out of it in terms of do they get to maintain the rule set MDE layer do they get to maintain the rule set because that's what we are communicating uh we need to let them know and come to an agreement saying hey does it does this match what you're wishing or what your business requirements are right you're

**[00:14:09] Harika**: asking for me when we need to engage them or

**[00:14:14] Harika**: are you saying that we need to engage them now before we even talk about this i'm

**[00:14:18] Shailendra**: not saying we involve them in writing the solution design i'm just i just want to make sure that some of the assumptions that we made for what what is standard between different hardware versus what is custom at least that high level assumption we should validate from from them i'm not saying yeah

**[00:14:41] Harika** ⚠: i think

**[00:14:44] Harika**: you're saying the same thing in different ways uh we are making assumptions yeah let's do make list

**[00:14:50] UNKNOWN** ?: List

**[00:14:50] Harika**: all the assumptions, build it, I'm not saying build the tool, build all your assumptions, build all the interactions, and then bring it to the business, say, hey,

**[00:14:59] Shailendra**: this

**[00:14:59] Harika** ⚠: is all we have assumed. Yeah, we're saying the same thing.

**[00:15:02] Shailendra**: Yeah, the assumption, we have to document them, and we have to run them by business before we finalize the solution design.

**[00:15:10] Uma**: Yes. Barika,

**[00:15:13] Uma**: I think based on what you are showing it here, maybe we can come up with the design roles right?

**[00:15:25] Uma**: Yes we need to. And maybe we can show

**[00:15:33] Harika**: those roles

**[00:15:34] Uma**: also to the users? We

**[00:15:36] Harika**: should.

**[00:15:41] Harika**: My point is if we keep engaging business for every few assumptions we make it will be too time consuming too much back and forth what what I was saying is let's build the models for SE Gateway from start to end, just for MVP, and then we engage the business saying, hey, these are all the assumptions, this is how your input we expect to look like, or this is how the interactions would look like between SE Meridian and this is the output of a request on MDE, right? We can present it once we model from end to end.

**[00:16:15] Shailendra**: Well, SE Gateway, yes, we can do that.

**[00:16:19] Shailendra**: more worried about the long term we

**[00:16:22] Shailendra**: have to then list out all assumptions for other hardware as well and then for that I think we have to go hardware by hardware like use case by use case so we

**[00:16:35] Harika**: already do it for the business, the same business you are referring to Shalindra, we do it hardware agnostic for them on Nessie side all

**[00:16:46] Harika**: Nessie activities are hardware agnostic so when we say we are building this layer to connection uh spec this is getting model off of what the information uma shared with me as well as how messy treats a layer to connection today which is already model hardware os agnostic and

**[00:17:09] Harika**: i'm saying that this is final because we are building it i'm saying let's build this and then we can get confirmation and when we have the conversations we can ask the questions hey does this look good enough for you or do you see anything missing as part of a user input.

**[00:17:27] Shailendra**: So those conversations beyond MVP you envision them to happen use case by use case?

**[00:17:36] Harika**: I don't know.

**[00:17:39] Shailendra**: That's one of the things.

**[00:17:43] Shailendra**: SE Gateway is fine. Either hard code or... But

**[00:17:49] Harika**: Shailendra, I don't know why you do not, why you are not convinced when I say that this is model agnostic at this moment. or

**[00:18:02] Shailendra**: you're talking about layer 3 as well and everything related to all the config layer 2 and entire layer 2 and layer 3 config nasi

**[00:18:14] Shailendra**: accomplishes today via service activities

**[00:18:17] Harika**: yeah they're all hardware agnostic and i'm telling you i don't know why you're not convinced

**[00:18:33] Shailendra**: but nasi service activities and whatever nasi needs or however nasi deploys the hardware config wise is all standardized house house

**[00:18:47] Kevin** ⚠: no no

**[00:18:48] Shailendra** ⚠: no

**[00:18:48] Kevin**: no no no what we're saying Schillinger is that what we need to you to send to Nessie is standardized we'll do we'll do the interpretations for different operating systems and that's why I said earlier you guys don't even worry about that just we just need to build the models for the layer 2 connection that we agree on can be be extracted from the information, the CIQ, the EPF, and built as a payload and sent to the Nessie site. Once it gets there, then we look at the devices, we look at operating systems, we go through our rules engine and decide if this is a Layer 2 connection and it's a port channel, what is that port channel called? What is the port channel ID for this design? What is the accepted VLAN and VRF arrangements, blah, blah, blah. So that's all specific to the devices that we're going to operate on. But for the purposes of what Harika is showing you here, it doesn't matter. It doesn't come into play yet.

**[00:19:49] Shailendra**: Yeah, I understand Nessie will handle all of that, and there's already months or years of history there. What I'm saying is it all starts at MDE in the new world. This is all going to start in the MDE and then the MDE and the orchestrator or a combination of them will still have to first come up with that first step to pick the right combination of CIQ and PIF and then have the right payloads sent down to Nessie, right? In order for you to then do all these things, all those things.

**[00:20:24] Kevin**: But the user will pick the CIQ and the PIF, right? for

**[00:20:33] Harika**: Nidhi I know we're running out of time and Suresh also has questions maybe Shalendra we can talk offline and

**[00:20:42] Nidhi**: just to maybe close that out Shalendra like how we model in MDE should be like an inventory model vendor agnostic layer so if we are building logical interfaces or layer 2 there is a layer 2 now for certain CIQs maybe they don't don't populate certain attributes and for others they do but it's still a layer two connection and these are everything available for you to make it so it has to get that object level

**[00:21:14] Nidhi**: and so and then we can build upon maybe that layer two doesn't go alone and we have to point the l3 inter ip interfaces to that and then build another service but those objects

**[00:21:26] Kevin** ⚠: should still

**[00:21:26] Nidhi**: be the same objects layer

**[00:21:28] Kevin**: two interfaces are in

**[00:21:31] Kevin**: four channels are industry standard ip3 ipv4 and bb6 all that stuff is industry standard there's nothing that we're using here in the models that harika is showing you that is not industry standard that

**[00:21:44] Kevin**: is the whole point use industry standard terminology so that regardless of what platform we're deploying on we'll be able to interpret it can

**[00:21:56] Uma**: you please share this i mean i know this is not the final one but can you just share it with me i will but it

**[00:22:03] Harika**: or

**[00:22:07] Uma**: maybe maybe next week or i'm bringing up the next week yeah i yeah just

**[00:22:12] Harika**: give me two two more days to um okay uh like normalize it with other i'm using examples from others to account for most of the fields for mb mbp and at least two more use cases beyond mbp once that is done give me two more days and i'll share it with you yeah

**[00:22:30] Nidhi**: and it still is very close to what we started from right yes

**[00:22:34] Harika**: yeah

**[00:22:34] Nidhi**: so i i think you're finalizing from an s perspective

**[00:22:38] Harika**: what all other

**[00:22:39] Nidhi**: things yeah

**[00:22:40] Harika**: but

**[00:22:41] Nidhi**: this is theoretically is exactly same right yes oh

**[00:22:47] Harika**: two things uh solution now you have a question maybe go ahead and ask i'll go to you yeah

**[00:22:54] Shailendra**: so uh if you go back to that previous blocks with respect to md meridian the

**[00:23:00] Harika**: nsc

**[00:23:01] Shailendra**: uh in the chart uh this is a channel so a way exactly to the l2 and l3 that they are coming

**[00:23:09] Harika**: here

**[00:23:10] Shailendra**: these

**[00:23:11] Harika**: things l2

**[00:23:13] Shailendra**: l3 yeah so now in the other page that you're showing right uh oh

**[00:23:18] Harika**: that should come from this uh network

**[00:23:22] Harika**: workflow orchestrated engine, right? This engine or the decision engine you have at the meridian layer would come up with those L2, L3 objects based on your CIQPF. All the data is coming

**[00:23:34] Nidhi**: from CIQPF. That's your inventory. It's how you are just building the payload for NASI in this workflow orchestration layer, but the data is still, you are not in the CIQPF. Yes.

**[00:23:47] Shailendra**: Is it at the logical resource assignment or at the network architecture based decision engine? So I think my assumption is that L2 and L3 that we are sending it to network architecture based decision engine. And

**[00:24:02] Uma** ⚠: the

**[00:24:02] Shailendra**: rules

**[00:24:03] Uma**: we will in

**[00:24:06] Uma**: the decision engine level that I think that we can create the payload that we will be sending to the NISI in the orchestrated way.

**[00:24:19] Shailendra**: and I mean like you know if in the payload in the L2 and L3 if we have more examples comments

**[00:24:33] Shailendra**: here

**[00:24:42] Harika**: so they're saying hey this comes from EPIP or this comes from CIQ like I intend to add comments in here so we can have a discussion but

**[00:24:54] Harika**: if any gaps this

**[00:24:58] Nidhi**: one other than for VLAN everything is the EPIP for L2

**[00:25:02] Harika**: yes correct yeah two

**[00:25:06] Harika**: things I want to bring up Nidhi one is when you look at this layer to connection then

**[00:25:15] Harika**: assumption I made discussed with Kevin but I want to bring it to the team begin

**[00:25:23] Harika** ?: to send Z

**[00:25:26] Harika**: side information into connection click give me the point the two points that you're connecting instead of providing information from a network element point of view this way Nessie can determine at the Nessie layer we can determine if we need to do configuration on A site and Z site or just Z site based on a shared data store between

**[00:25:52] Harika**: Meridian and Meridian MDE and NSE. This doesn't exist today. This needs to be built.

**[00:26:01] Harika**: We need to know which devices are for packet code domain, which devices are for IP core, which would be IP transport, IP vendor, more from Meridian layer. And once that information is passed down to us or once we have the shared data store we can at the Nessie layer we will decide if we need to do that L2 ether channel add and L2 interface add on both devices of one because this this will be eventually used for interest IP to IP connections right from these

**[00:26:33] Harika**: two could be network devices that

**[00:26:38] Harika**: scenario Nessie needs to configure both ends.

**[00:26:41] Uma**: So Nidhi do you think we can get that details I mean if it is the packet core or? I

**[00:26:48] Nidhi**: was going to say how are we thinking of doing that because we do have like type of the shelf in EI but some of them are very generic so we need to go back and check where we can like why do we want to maintain in a different layer I think if it needs to be identified they need to identify when the database the device in EI right so that it's it's it's for everyone to consume yeah

**[00:27:15] Harika**: if If it is part of existing data stores like EAI or CMDB, then we can use them. I'm just saying that this logic needs to exist somewhere or this

**[00:27:29] Uma** ⚠: mapping needs

**[00:27:29] Harika**: to exist somewhere.

**[00:27:30] Uma**: So, Nidhi, do you think the EAI will give me the demand and differences?

**[00:27:36] Nidhi** ⚠: It's

**[00:27:37] Uma**: not

**[00:27:37] Nidhi**: going to be straightforward. First of all, who has this holistic list of all routers and all these things and they can quantify this? is like we can say it rather that okay pcf it's a packet core but we need to make sure that we have data points in ei that can reflect it it is currently generically database right like in the sense a lot of types are router types then you have different there's several different types in ei but how they are choosing we need to go back and analyze it for mvp yeah you can because you know this is my yeah you can do that but if you are looking for future ahead then we need to make sure we have a sustainable way for for identifying that and

**[00:28:25] Shailendra**: that sustainable way is what i was my slides are about are you guys uh have another meeting or oh

**[00:28:32] Nidhi**: yeah yeah

**[00:28:33] Shailendra**: okay

**[00:28:36] Shailendra** ?: yeah

**[00:28:38] Nidhi** ?: yeah we can maybe do

**[00:28:40] Shailendra**: you have a few minutes now nidhi i can maybe walk in harika later individually about the slides let

**[00:28:48] Nidhi**: me check give me one second i know i had like a long big call today but give me one second i

**[00:28:56] Nidhi**: mean i can stay five if it's five ten minutes that is fine i think i can

**[00:28:59] Shailendra**: yeah i'll try to make it quick and then

**[00:29:02] Nidhi**: other things sorry i was gonna go like yes harika is looking for this is bundle it's not evident from etf so we will have to talk back because it's all trains four trains we know by the design it is bundling but we don't know when they're filling the epec so we need to make sure we can and also i would add the trail ids as well harika so that we give you full set of data if we are troubleshooting and things like those we can go back and check because that really ties the ports at both ends so like if you don't mind sending the json that would be great so that we can see if there is something else that was missing seeing from previous iterations and others yeah

**[00:29:44] Harika** ?: yeah

**[00:29:47] Harika**: give me one more day Nidhi yeah it's

**[00:29:49] Nidhi** ⚠: okay not gonna look on weekend so okay

**[00:29:53] Shailendra**: so Harika can you also stay or you need to drop I

**[00:29:58] Harika**: need to drop for the Kafka okay

**[00:29:59] Shailendra**: I'll ping you separately I do need to walk you through my slides as well I know you had some opinions but I think it would be best if you talk live yep


## Talk time

- Harika: 11.3 min
- Shailendra: 5.7 min
- Nidhi: 5.1 min
- Kevin: 2.2 min
- Uma: 1.1 min
- UNKNOWN: 0.1 min

## Stage 6 — Download results (also saved in your Drive under `OUT`)

In [23]:
from google.colab import files
for f in ["focus_refined.md", "focus_refined.csv", "focus_refined.srt"]:
    files.download(f"{fdir}/{f}")
files.download(f"{OUT}/transcript.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# Make an MP4: meeting audio + speaker-labeled subtitles burned in
srt = f"{fdir}/focus_refined.srt"
mp4 = f"{WORK}/MDE_Design_Freeze_with_speakers.mp4"

!ffmpeg -y -hide_banner -loglevel error \
  -f lavfi -i color=c=black:s=1280x720:r=5 -i "{AUDIO}" \
  -vf "subtitles={srt}:force_style='FontSize=22,Alignment=2,MarginV=40'" \
  -c:v libx264 -preset veryfast -tune stillimage -c:a aac -b:a 128k -shortest "{mp4}"

from google.colab import files
files.download(mp4)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>